# SOV7 LoRA Training - Mistral-7B

Fine-tune Mistral-7B-Instruct-v0.3 with QLoRA on sovereign AI training data (10,906 pairs).

**Runtime:** T4 GPU (16GB VRAM)  
**Method:** QLoRA (4-bit NF4 + LoRA r=32)  
**Data:** 10,906 training pairs across 44 families

In [ ]:
# Install dependencies
!pip install -q transformers peft trl bitsandbytes accelerate datasets

In [ ]:
# Download training data from GitHub
!wget -q https://raw.githubusercontent.com/CSOAI-ORG/csoai-static-deploy2/main/training_data/honey_mistral.jsonl
!wc -l honey_mistral.jsonl
!head -2 honey_mistral.jsonl

In [ ]:
import json
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Load data
records = []
for line in open('honey_mistral.jsonl'):
    try:
        d = json.loads(line)
        q, a = d['q'].strip(), d['a'].strip()
        if q and a:
            records.append({'text': f'<s>[INST] {q} [/INST] {a}</s>'})
    except:
        continue

print(f'Loaded {len(records)} examples')
dataset = Dataset.from_list(records)

In [ ]:
# Load model with QLoRA
BASE = 'mistralai/Mistral-7B-Instruct-v0.3'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)
print('Model loaded!')

In [ ]:
# Apply LoRA
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Train
OUTPUT_DIR = '/content/lora_sov7'

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy='epoch',
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    max_seq_length=1024,
    report_to=[],
    seed=42,
    dataset_text_field='text',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

started = time.time()
trainer.train()
elapsed = time.time() - started
print(f'Training done in {elapsed/60:.1f} min')

In [ ]:
# Save adapter
model.save_pretrained(f'{OUTPUT_DIR}/final')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/final')
print(f'Saved adapter to {OUTPUT_DIR}/final')

# Zip and download
!cd {OUTPUT_DIR}/final && zip -r /content/lora_adapter.zip .
from google.colab import files
files.download('/content/lora_adapter.zip')